# Tome 14, Chapter 9: Chebyshev's bias (Python companion)

Exercises first, solutions at the end. We run the race between the primes $4n+1$ and $4n+3$ to $10^8$, compute zeros of $L(s,\chi_{-4})$, compare the race with its model $1+\sum_\gamma x^{i\gamma}/\rho$, compute the variance $V$ two ways, and recover Rubinstein and Sarnak's $\delta(4;3,1)=0.9959$. The computer science application: **the segmented sieve of Eratosthenes**, which finds all primes up to $N$ in memory $O(\sqrt N)$ instead of $O(N)$. Requires `numpy` and `mpmath`.

In [ ]:
import math, time
import numpy as np
import mpmath

## Exercises

**P9.1 (★, CS application) A segmented sieve.** Write a function that yields the primes in $[lo,hi)$ for consecutive blocks of length $10^7$, using only the primes up to $\sqrt N$. Run the race to $N=10^8$: record $\pi(N;4,1)$, $\pi(N;4,3)$, the first $x$ where $\pi(x;4,1)>\pi(x;4,3)$, the intervals of primes where the form $4n+1$ leads, and the share of $\log x\in[\log3,\log N]$ during which it leads.

**P9.2 (★★) Zeros of $L(s,\chi_{-4})$.** The function $Z(t)=e^{i\vartheta(t)}L(\frac12+it,\chi_{-4})$, with $\vartheta(t)=\operatorname{Im}\log\Gamma(\frac34+\frac{it}2)+\frac t2\log\frac4\pi$, is real (by the functional equation with root number $1$). Find its sign changes on $[0,400]$ with step $0.1$, refine them with `mpmath.findroot`, and compare the number found with $\frac12N(T)$, $N(T)\approx\frac T\pi\log\frac{4T}{2\pi e}$.

**P9.3 (★★) The race and its model.** On a grid of $u=\log x$ in $[10,\log10^8]$, compare $E(u)=\frac{u}{e^{u/2}}(\pi(e^u;4,3)-\pi(e^u;4,1))$ with $E_T(u)=1+2\sum_{0<\gamma<T}\operatorname{Re}\frac{e^{i\gamma u}}{\frac12+i\gamma}$ for $T=50,100,200,400$: compute the mean square difference (Proposition 9.4).

**P9.4 (★★) The variance two ways.** Compute $\sum_{0<\gamma<400}\frac2{\frac14+\gamma^2}$, add the tail estimated with the density $\frac1{2\pi}\log\frac{4t}{2\pi}$ of the ordinates, and compare with $V=\log\frac4\pi-\gamma+2\frac{L'}L(1,\chi_{-4})$ (Exercise 9.4).

**P9.5 (★★★) The bias, computed.** Compute $\delta(4;3,1)=\frac12+\frac1\pi\int_0^\infty\frac{\sin\xi}\xi\prod_{\gamma>0}J_0(\frac{2\xi}{|\rho|})\,\mathrm d\xi$ with the zeros of P9.2, replacing the zeros above $400$ by a Gaussian with the tail variance of P9.4. Compare with the empirical share of P9.1 and with Chebyshev's bound $1-\frac V2$.

**P9.6 (★) Chebyshev's sum.** Chebyshev asserted that $\sum_{p>2}(-1)^{(p-1)/2}e^{-p/X}\to-\infty$; Hardy, Littlewood and Landau showed that this is equivalent to GRH for $\chi_{-4}$. Compute $S(X)=\sum_{p>2}\chi_{-4}(p)\log p\,e^{-p/X}$ for $X=10^2,\dots,10^7$ and compare with $-\frac12\sqrt{\pi X}$, the contribution of the prime squares.

## Solutions

### P9.1

In [ ]:
def small_primes(n):
    s = np.ones(n + 1, dtype=bool); s[:2] = False
    for i in range(2, int(n ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

def segmented_primes(N, block=10 ** 7):
    base = small_primes(int(N ** 0.5) + 1)
    for lo in range(0, N, block):
        hi = min(lo + block, N)
        s = np.ones(hi - lo, dtype=bool)
        for p in base:
            if p * p >= hi: break
            start = max(p * p, -(-lo // p) * p)
            s[start - lo::p] = False
        if lo == 0: s[:2] = False
        yield np.nonzero(s)[0] + lo

N = 10 ** 8
c1 = c3 = 0; first = None; lead_log = 0.0; leads = []; t0 = time.time()
for ps in segmented_primes(N):
    ps = ps[ps > 2]
    sgn = np.where(ps % 4 == 1, -1, 1)          # +1 for 4n+3, -1 for 4n+1
    diff = np.cumsum(sgn) + (c3 - c1)           # pi(x;4,3) - pi(x;4,1) just after each prime
    nxt = np.append(ps[1:], ps[-1] + 1)
    neg = diff < 0
    if neg.any():
        lead_log += float(np.sum(np.log(nxt[neg] / ps[neg])))
        if first is None: first = int(ps[np.argmax(neg)])
        # starts and ends of leading stretches
        edges = np.flatnonzero(np.diff(np.concatenate([[0], neg.astype(int), [0]])))
        for a, b in zip(edges[::2], edges[1::2]):
            leads.append((int(ps[a]), int(nxt[b - 1])))
    c1 += int(np.sum(ps % 4 == 1)); c3 += int(np.sum(ps % 4 == 3))
print(f"pi(N;4,1) = {c1}, pi(N;4,3) = {c3}, first lead of 4n+1 at {first}, {time.time() - t0:.1f}s")
merged = []
for a, b in leads:
    if merged and a <= merged[-1][1]: merged[-1] = (merged[-1][0], b)
    else: merged.append((a, b))
print("stretches where 4n+1 leads (merged):", len(merged))
for a, b in merged[:12]: print("  ", a, "-", b)
print("share of log scale:", lead_log / math.log(N / 3))

The counts are $\pi(10^8;4,1)=2\,880\,504$ and $\pi(10^8;4,3)=2\,880\,950$, and the first lead change is at $26861$, as Leech found. The form $4n+1$ leads only in a few clusters of short stretches (at $26861$, then from $616841$ on, and later ones), for about $0.045\%$ of the logarithmic scale. The memory used is one block of $10^7$ booleans plus the $1229$ primes below $10^4$, whatever $N$ is: this is why segmented sieves are used for every large prime count.

### P9.2

In [ ]:
mpmath.mp.dps = 15
def Lchi(s):
    return 4 ** (-s) * (mpmath.zeta(s, 0.25) - mpmath.zeta(s, 0.75))

def vartheta(t):
    return mpmath.im(mpmath.loggamma(mpmath.mpf(0.75) + 0.5j * t)) + t / 2 * mpmath.log(4 / mpmath.pi)

def Zchi(t):
    return mpmath.re(mpmath.exp(1j * vartheta(t)) * Lchi(mpmath.mpf(0.5) + 1j * t))

t0 = time.time()
grid = np.arange(0.1, 400, 0.1)
vals = np.array([float(Zchi(t)) for t in grid])
zeros = []
for i in np.flatnonzero(vals[:-1] * vals[1:] < 0):
    zeros.append(float(mpmath.findroot(Zchi, (grid[i], grid[i + 1]), solver='anderson')))
zeros = np.array(zeros)
NT = lambda T: T / math.pi * math.log(4 * T / (2 * math.pi * math.e)) + 0.75
print(f"{len(zeros)} zeros in (0,400), expected about {NT(400) / 2:.1f}; {time.time() - t0:.0f}s")
print("first five:", np.round(zeros[:5], 4))

The first zeros are $6.0209$, $10.2438$, $12.9881$, $16.3426$, $18.2920$, and the count agrees with $\frac12N(400)$ up to the bounded error term $S(T)$. (A sign change can only miss a pair of very close zeros; the count shows that none was missed.)

### P9.3

In [ ]:
# pi(x;4,3)-pi(x;4,1) on a log grid, from a second pass of the sieve
us = np.linspace(10, math.log(N) - 0.01, 3000)
xs = np.exp(us); E = np.zeros_like(us); j = 0; c1 = c3 = 0
for ps in segmented_primes(N):
    ps = ps[ps > 2]
    diff = np.cumsum(np.where(ps % 4 == 1, -1, 1)) + (c3 - c1)
    while j < len(xs) and xs[j] < ps[-1]:
        k = np.searchsorted(ps, xs[j], side='right') - 1
        d = diff[k] if k >= 0 else c3 - c1
        E[j] = d * us[j] / math.sqrt(xs[j]); j += 1
    c1 += int(np.sum(ps % 4 == 1)); c3 += int(np.sum(ps % 4 == 3))

def E_model(u, T):
    g = zeros[zeros < T]
    return 1 + 2 * np.real(np.exp(1j * np.outer(u, g)) / (0.5 + 1j * g)).sum(axis=1)

for T in [50, 100, 200, 400]:
    print(f"T={T}: mean square of E - E_T = {np.mean((E - E_model(us, T)) ** 2):.4f}")
print("mean of E:", E.mean(), " mean square of E - 1:", np.mean((E - 1) ** 2))

The mean square difference decreases with $T$ (from $0.066$ to $0.042$) but stays well above the $\frac{\log^3T}T$ of Proposition 9.4: on the short range $u\le18.4$ the lower-order terms, of relative size $\frac1u$, still matter, notably the term $G$ of the proof and the error in $\theta(\sqrt x)$. For the same reason the mean of $E$ over this range is $1.19$ rather than $1$.

### P9.4

In [ ]:
gamma_E = float(mpmath.euler)
head = np.sum(2 / (0.25 + zeros ** 2))
T0 = 400
tail = float(mpmath.quad(lambda t: 2 / t ** 2 * mpmath.log(4 * t / (2 * mpmath.pi)) / (2 * mpmath.pi), [T0, mpmath.inf]))
dL = mpmath.diff(Lchi, 1 + mpmath.mpf('1e-12'))
V_formula = math.log(4 / math.pi) - gamma_E + 2 * float(dL) / (math.pi / 4)
print(f"zeros below 400: {head:.5f}; tail estimate: {tail:.5f}; total {head + tail:.5f}")
print(f"formula: V = {V_formula:.5f}")

The two values agree to about four decimals: $V=0.15557$. The first zero alone contributes $\frac2{\frac14+6.02^2}=0.055$, more than a third of the total.

### P9.5

In [ ]:
from scipy.special import j0
from scipy.integrate import quad

amp = 2 / np.sqrt(0.25 + zeros ** 2)          # amplitudes 2/|rho|
tail_var = tail

def phi(xi):
    return np.prod(j0(amp * xi)) * math.exp(-tail_var * xi * xi / 2)

# the integrand decays fast; integrate on [0, 60] in pieces
total = 0.0
for a in np.arange(0, 60, 0.5):
    total += quad(lambda x: math.sin(x) / x * phi(x) if x > 0 else phi(0), a, a + 0.5, limit=200)[0]
delta = 0.5 + total / math.pi
print(f"delta(4;3,1) = {delta:.5f}")
print(f"Chebyshev bound 1 - V/2 = {1 - V_formula / 2:.4f}")

The integral gives $\delta(4;3,1)\approx0.9959$, Rubinstein and Sarnak's value. Chebyshev's inequality only gave $0.922$. The empirical share up to $10^8$ in P9.1, about $0.9996$, is larger than $\delta$: the race has not yet explored its whole distribution, since $\log 10^8\approx18$ is a short stretch of the variable $u$ compared with the periods of the many waves that must line up for class $1$ to lead.

### P9.6

In [ ]:
ps = small_primes(10 ** 8)
ps = ps[ps > 2]
chi = np.where(ps % 4 == 1, 1.0, -1.0)
lg = np.log(ps)
for X in [1e2, 1e3, 1e4, 1e5, 1e6, 1e7]:
    m = ps < 40 * X
    S = np.sum(chi[m] * lg[m] * np.exp(-ps[m] / X))
    S0 = np.sum(chi[m] * np.exp(-ps[m] / X))
    print(f"X={X:.0e}: S = {S:10.2f},  -sqrt(pi X)/2 = {-math.sqrt(math.pi * X) / 2:10.2f},  unweighted sum = {S0:8.2f}")

$S(X)$ follows $-\frac12\sqrt{\pi X}$, with a relative gap that shrinks from $37\%$ at $X=10^2$ to $1\%$ at $X=10^7$; the gap comes from lower-order terms such as the cubes of primes and the error in the prime number theorem for the squares. Under GRH, $\sum_n\chi(n)\Lambda(n)e^{-n/X}=-\sum_\rho\Gamma(\rho)X^\rho+O(1)$, and $|\Gamma(\frac12+i\gamma)|=\sqrt{\pi/\cosh\pi\gamma}$ is about $10^{-4}$ at the first zero: the waves are damped almost to nothing, and only the squares remain. That is why the smoothed sum is so much more regular than the race itself, and why Chebyshev, who looked at such sums, was so confident. The unweighted sum, his original one, also decreases steadily.